In [ ]:
import json
import aiohttp
import websockets

FEE = 0.0002
ROUND_TRIP_FEE = 2 * FEE
EXCHANGE_INFO_URL = "https://api.binance.com/api/v3/exchangeInfo"
WS_BASE = "wss://stream.binance.com:9443/stream?streams="

async def get_symbols():

    async with aiohttp.ClientSession() as session:

        async with session.get(EXCHANGE_INFO_URL) as response:

            data = await response.json()

    symbols = []

    for s in data["symbols"]:

        # Only currently trading spot markets
        if s["status"] != "TRADING":
            continue

        # Only USDT pairs for now
        if s["quoteAsset"] != "USDT":
            continue

        symbols.append(s["symbol"].lower())

    return symbols


async def scanner():
    symbols = await get_symbols()

    print(f"Found {len(symbols)} USDT markets")

    streams = "/".join(
        f"{symbol}@bookTicker"
        for symbol in symbols
    )

    url = WS_BASE + streams

    print("Connecting to Binance...")

    async with websockets.connect(url, ping_interval=20, ping_timeout=20, max_size=None) as ws:
        print("Connected!")
        print(f"Minimum spread: {ROUND_TRIP_FEE * 100:.4f}%")
        print()

        while True:
            message = await ws.recv()
            data = json.loads(message)

            ticker = data["data"]

            symbol = ticker["s"]
            bid = float(ticker["b"])
            ask = float(ticker["a"])

            bid_qty = float(ticker["B"])
            ask_qty = float(ticker["A"])

            bid_liquidity = bid * bid_qty
            ask_liquidity = ask * ask_qty

            MIN_LIQUIDITY = 1000

            if bid <= 0 or ask <= 0:
                continue

            # Spread relative to bid
            gross_spread = (ask - bid) / bid

            # Net after 2 sides of fees
            net_spread = gross_spread - ROUND_TRIP_FEE

            if(
                net_spread > 0
                and bid_liquidity >= MIN_LIQUIDITY
                and ask_liquidity >= MIN_LIQUIDITY
            ):

                print(
                    f"{symbol:12} "
                    f"Spread: {gross_spread * 100:.4f}% "
                    f"Net: {net_spread * 100:.4f}% "
                    f"Bid Liq: ${bid_liquidity:,.2f} "
                    f"Ask Liq: ${ask_liquidity:,.2f}"
                )

await scanner()

Found 485 USDT markets
Connecting to Binance...
Connected!
Minimum spread: 0.0400%

SHIBUSDT     Spread: 0.1894% Net: 0.1494% Bid Liq: $28,780.37 Ask Liq: $20,756.73
ADAUSDT      Spread: 0.0479% Net: 0.0079% Bid Liq: $1,494.53 Ask Liq: $11,710.69
ADAUSDT      Spread: 0.0479% Net: 0.0079% Bid Liq: $1,494.53 Ask Liq: $11,510.67
ADAUSDT      Spread: 0.0479% Net: 0.0079% Bid Liq: $1,494.53 Ask Liq: $11,310.66
ADAUSDT      Spread: 0.0479% Net: 0.0079% Bid Liq: $1,494.53 Ask Liq: $11,110.65
PYTHUSDT     Spread: 0.0424% Net: 0.0024% Bid Liq: $1,947.61 Ask Liq: $1,942.75
PARTIUSDT    Spread: 0.4184% Net: 0.3784% Bid Liq: $7,677.47 Ask Liq: $6,633.64
PEPEUSDT     Spread: 0.2653% Net: 0.2253% Bid Liq: $197,206.42 Ask Liq: $178,276.37
PEPEUSDT     Spread: 0.2653% Net: 0.2253% Bid Liq: $197,227.09 Ask Liq: $178,276.37
PEPEUSDT     Spread: 0.2653% Net: 0.2253% Bid Liq: $197,247.75 Ask Liq: $178,276.37
PEPEUSDT     Spread: 0.2653% Net: 0.2253% Bid Liq: $197,268.41 Ask Liq: $178,276.37
WLFIUSDT     S

CancelledError: 

In [ ]:
# Best-looking from your snapshot
# Market	Net spread	Bid Liq	Ask Liq	My take
# PARTIUSDT	0.3802%	$7.6k	$1.0k	🔥 High edge, but ask-side liquidity is thin
# ACHUSDT	0.3529%	$4.2k	$3.2k	🔥 Very interesting — relatively balanced
# CYBERUSDT	0.2725%	$4.2k	$3.0k	🔥 Good balance + decent edge
# KSMUSDT	0.2433%	$4.3k	$1.2k	Good spread, but ask liquidity weak
# PEPEUSDT	0.2260%	$206k	$34k	⭐ Excellent depth; strongest liquidity candidate
# SKLUSDT	0.2211%	$3.5k	$1.5k	Interesting, but relatively shallow
# GMXUSDT	0.2263%	$1.1k	$1.3k	Good balance, but very small absolute depth
# SHIBUSDT	0.1494%	$11k	$21k	⭐ Good depth and reasonable edge
# APTUSDT	0.1379%	$11.2k	~$7–10k	⭐ Nice balanced MM candidate
# BNSOLUSDT	0.1351%	$2.4–3.1k	$14k	Edge okay, but strongly ask-heavy
# My shortlist

# 1. PEPE — probably the most interesting if you care about actually deploying meaningful size. The 0.226% net is attractive and the displayed liquidity is dramatically higher than most of the others. The caveat is the very asymmetric $206k bid vs ~$34k ask.

# 2. ACH — probably my favorite balanced opportunity. 0.353% net with roughly $4.2k / $3.2k liquidity is much healthier than something showing $7k bid and $1k ask.

# 3. CYBER — similar logic: 0.273% net, ~$4.2k bid / ~$3.0k ask. Good combination of edge and balance.

# 4. APT — lower edge at 0.138%, but considerably more convincing as a market to continuously quote because liquidity is reasonably balanced and ~$9–11k is available on the bid.

# 5. SHIB — 0.149% net with ~$11k / $21k displayed. The spread isn't spectacular, but the depth makes it more practical.

In [1]:
import json
import aiohttp
import websockets

FEE = 0.001
ROUND_TRIP_FEE = 2 * FEE
EXCHANGE_INFO_URL = "https://api.binance.com/api/v3/exchangeInfo"
WS_BASE = "wss://stream.binance.com:9443/stream?streams="

async def get_symbols():

    async with aiohttp.ClientSession() as session:

        async with session.get(EXCHANGE_INFO_URL) as response:

            data = await response.json()

    symbols = []

    for s in data["symbols"]:

        # Only currently trading spot markets
        if s["status"] != "TRADING":
            continue

        # Only USDT pairs for now
        if s["quoteAsset"] != "USDT":
            continue

        symbols.append(s["symbol"].lower())

    return symbols


async def scanner():
    symbols = await get_symbols()

    print(f"Found {len(symbols)} USDT markets")

    streams = "/".join(
        f"{symbol}@bookTicker"
        for symbol in symbols
    )

    url = WS_BASE + streams

    print("Connecting to Binance...")

    async with websockets.connect(url, ping_interval=20, ping_timeout=20, max_size=None) as ws:
        print("Connected!")
        print(f"Minimum spread: {ROUND_TRIP_FEE * 100:.4f}%")
        print()

        while True:
            message = await ws.recv()
            data = json.loads(message)

            ticker = data["data"]

            symbol = ticker["s"]
            bid = float(ticker["b"])
            ask = float(ticker["a"])

            bid_qty = float(ticker["B"])
            ask_qty = float(ticker["A"])

            bid_liquidity = bid * bid_qty
            ask_liquidity = ask * ask_qty

            MIN_LIQUIDITY = 1000

            if bid <= 0 or ask <= 0:
                continue

            # Spread relative to bid
            gross_spread = (ask - bid) / bid

            # Net after 2 sides of fees
            net_spread = gross_spread - ROUND_TRIP_FEE

            if(
                net_spread > 0
                and bid_liquidity >= MIN_LIQUIDITY
                and ask_liquidity >= MIN_LIQUIDITY
            ):

                print(
                    f"{symbol:12} "
                    f"Spread: {gross_spread * 100:.4f}% "
                    f"Net: {net_spread * 100:.4f}% "
                    f"Bid Liq: ${bid_liquidity:,.2f} "
                    f"Ask Liq: ${ask_liquidity:,.2f}"
                )

await scanner()

Found 486 USDT markets
Connecting to Binance...
Connected!
Minimum spread: 0.2000%

LUNAUSDT     Spread: 0.4405% Net: 0.2405% Bid Liq: $3,780.59 Ask Liq: $4,547.80
LUNAUSDT     Spread: 0.4405% Net: 0.2405% Bid Liq: $3,780.59 Ask Liq: $4,542.69
LUNAUSDT     Spread: 0.4405% Net: 0.2405% Bid Liq: $3,780.59 Ask Liq: $4,537.58
TOWNSUSDT    Spread: 0.4464% Net: 0.2464% Bid Liq: $3,427.61 Ask Liq: $1,492.45
TOWNSUSDT    Spread: 0.4464% Net: 0.2464% Bid Liq: $3,427.61 Ask Liq: $1,548.70
BONKUSDT     Spread: 0.3175% Net: 0.1175% Bid Liq: $12,831.99 Ask Liq: $16,348.31
BONKUSDT     Spread: 0.3175% Net: 0.1175% Bid Liq: $12,831.99 Ask Liq: $17,630.67
PEPEUSDT     Spread: 0.2732% Net: 0.0732% Bid Liq: $174,012.75 Ask Liq: $178,628.95
PEPEUSDT     Spread: 0.2732% Net: 0.0732% Bid Liq: $174,012.75 Ask Liq: $179,977.43
PARTIUSDT    Spread: 0.4049% Net: 0.2049% Bid Liq: $6,155.42 Ask Liq: $4,536.05
PARTIUSDT    Spread: 0.4049% Net: 0.2049% Bid Liq: $6,155.42 Ask Liq: $4,595.44
PARTIUSDT    Spread: 0.4

CancelledError: 

In [1]:
import json
import aiohttp
import websockets

FEE = 0.0002                 # example: 0.02% per side
ROUND_TRIP_FEE = 2 * FEE

EXCHANGE_INFO_URL = "https://fapi.binance.com/fapi/v1/exchangeInfo"
WS_BASE = "wss://fstream.binance.com/stream?streams="

async def get_symbols():

    async with aiohttp.ClientSession() as session:

        async with session.get(EXCHANGE_INFO_URL) as response:
            data = await response.json()

    symbols = []

    for s in data["symbols"]:

        # Only perpetual contracts
        if s["contractType"] != "PERPETUAL":
            continue

        # Only currently trading
        if s["status"] != "TRADING":
            continue

        # USDⓈ-M / USDT margined
        if s["quoteAsset"] != "USDT":
            continue

        symbols.append(s["symbol"].lower())

    return symbols


async def scanner():

    symbols = await get_symbols()

    print(f"Found {len(symbols)} USDT perpetuals")

    streams = "/".join(
        f"{symbol}@bookTicker"
        for symbol in symbols
    )

    url = WS_BASE + streams

    print("Connecting to Binance USD-M Futures...")

    async with websockets.connect(
        url,
        ping_interval=20,
        ping_timeout=20,
        max_size=None
    ) as ws:

        print("Connected!")
        print(f"Minimum spread: {ROUND_TRIP_FEE * 100:.4f}%")
        print()

        while True:

            message = await ws.recv()
            data = json.loads(message)

            ticker = data["data"]

            symbol = ticker["s"]

            bid = float(ticker["b"])
            ask = float(ticker["a"])

            bid_qty = float(ticker["B"])
            ask_qty = float(ticker["A"])

            if bid <= 0 or ask <= 0:
                continue

            gross_spread = (ask - bid) / bid

            net_spread = gross_spread - ROUND_TRIP_FEE

            # USD value at top of book
            bid_liquidity = bid * bid_qty
            ask_liquidity = ask * ask_qty

            MIN_LIQUIDITY = 1000

            if (
                net_spread > 0
                and bid_liquidity >= MIN_LIQUIDITY
                and ask_liquidity >= MIN_LIQUIDITY
            ):

                print(
                    f"{symbol:16} "
                    f"Spread: {gross_spread * 100:.4f}% "
                    f"Net: {net_spread * 100:.4f}% "
                    f"Bid Liq: ${bid_liquidity:,.2f} "
                    f"Ask Liq: ${ask_liquidity:,.2f}"
                )


await scanner()

Found 526 USDT perpetuals
Connecting to Binance USD-M Futures...
Connected!
Minimum spread: 0.0400%

TUTUSDT          Spread: 0.0840% Net: 0.0440% Bid Liq: $1,104.32 Ask Liq: $1,128.20
ADAUSDT          Spread: 0.0448% Net: 0.0048% Bid Liq: $48,728.85 Ask Liq: $48,091.44
TRUMPUSDT        Spread: 0.0419% Net: 0.0019% Bid Liq: $12,878.29 Ask Liq: $21,443.24
TRUMPUSDT        Spread: 0.0419% Net: 0.0019% Bid Liq: $12,878.29 Ask Liq: $21,476.67
TRUMPUSDT        Spread: 0.0419% Net: 0.0019% Bid Liq: $12,878.29 Ask Liq: $21,510.10
TRUMPUSDT        Spread: 0.0419% Net: 0.0019% Bid Liq: $12,878.29 Ask Liq: $21,543.53
TRUMPUSDT        Spread: 0.0419% Net: 0.0019% Bid Liq: $12,878.29 Ask Liq: $21,574.58
TRUMPUSDT        Spread: 0.0419% Net: 0.0019% Bid Liq: $12,878.29 Ask Liq: $21,615.17
TRUMPUSDT        Spread: 0.0419% Net: 0.0019% Bid Liq: $12,878.29 Ask Liq: $21,650.99
TRUMPUSDT        Spread: 0.0419% Net: 0.0019% Bid Liq: $12,878.29 Ask Liq: $21,686.81
TRUMPUSDT        Spread: 0.0419% Net: 0.0

ConnectionClosedError: no close frame received or sent

Assuming **0.001 = 0.1% fee per side**:

 - Bid: **0.0000035000**
- Ask: **0.0000035100**
- Raw spread: **0.0000000100**
- Mid: **0.0000035050**
- Raw spread %:\

  $$
  \frac{0.0000035100-0.0000035000}{0.0000035050}
    \approx 0.2853\%
  $$

 If you **buy at the bid and later sell at the ask**, you pay the fee on both transactions:

 $$
\text{Net PnL} = Ask(1-0.001)-Bid(1+0.001)
$$

 $$
=0.0000035100(0.999)-0.0000035000(1.001)
$$

 $$
=0.00000350649-0.00000350350
=\boxed{0.00000000299}
$$

 So your **net spread capture is \~0.0854% of the mid price**.

 ### Important takeaway

 Your **0.2853% gross spread** gets eaten by roughly **0.2% in round-trip fees**, leaving only:

 **≈ 0.0854% net spread capture.**

 For example, on **$10,000 notional**, that's approximately **$8.54 net**, assuming both fills happen at those quoted prices and there are no other costs/slippage.